# ShopKart Order Profitability Prediction
Predict whether an order will generate **High Profit** or **Low Profit** before dispatch, so ShopKart can flag low-profit orders, tune pricing/discount strategy, and reduce shipping losses.

**Fixes applied to the original notebook are called out in `# FIX:` comments throughout.**
Key bugs that were corrected:
1. Model dict was defined *after* the cell that used it (`NameError`) — the two cells are now merged/reordered.
2. `Category`/`Gender` label-encoding was assigned with `.loc[:, col] = ...`, which silently kept the column as `object` dtype instead of `int` (confirmed in the original notebook's own cached output) — fixed with direct column assignment + `.astype(int)`.
3. Outlier removal (`remove_outliers_iqr`) produced `df_clean` but it was never used — the rest of the pipeline kept using the un-cleaned `df`. Now the cleaned frame is the one carried forward.
4. Obviously invalid raw values (negative `Qty`, negative `Discount`, `Customer_Age` of 0 or 150) were never corrected, despite being explicitly required. Added.
5. `StandardScaler`/`MinMaxScaler` were fit on the *entire* dataset before the train/test split (data leakage) and the scaled arrays were never actually used by any model. Scaling is now fit on the training split only and applied consistently.
6. A duplicate/dead `train_test_split` cell existed before the real feature set (`X`) was finalized, using `Order_ID`/`Order_Date` as features — removed.
7. Missing EDA pieces required by the spec (histograms, count plots) added.
8. Missing pipeline steps required by the spec added: Step 13 (overfitting/underfitting comparison across all models), feature importance for all 3 tree-based models (not just the tuned RF), a full preprocessing pipeline saved with the model so Step 17 (predict new data) actually works, and Steps 18–20 (Streamlit app, requirements.txt, deployment instructions).


## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, classification_report,
                              ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
import joblib

sns.set(style="whitegrid")
RANDOM_STATE = 42


## Step 2: Load Dataset

In [ ]:
df = pd.read_csv("shopkart_sales_dataset.csv")

print("First rows:")
display(df.head())

print("Last rows:")
display(df.tail())

print("Shape:", df.shape)


## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
df.info()


In [ ]:
df.dtypes


In [ ]:
print("Missing values per column:")
df.isnull().sum()


In [ ]:
print("Duplicate rows:", df.duplicated().sum())


In [ ]:
df.describe(include='all')


In [ ]:
# Value counts for key categorical columns
for col in ['Gender', 'City', 'Category']:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()


In [ ]:
# Class balance of the target
print(df['Profit_Category'].value_counts())
df['Profit_Category'].value_counts(normalize=True).plot(kind='bar', title='Profit_Category class balance')
plt.ylabel('Proportion')
plt.show()


In [ ]:
# Correlation matrix (numeric columns only)
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()


In [ ]:
# FIX: histograms were required by the spec but missing from the original notebook
num_cols_for_hist = ['Customer_Age', 'Qty', 'Unit Price', 'Discount', 'Shipping',
                      'Delivery', 'Sales', 'Profit', 'Rating']
df[num_cols_for_hist].hist(figsize=(14, 10), bins=30)
plt.suptitle("Histograms of Numeric Features")
plt.tight_layout()
plt.show()


In [ ]:
# FIX: count plots were required by the spec but missing from the original notebook
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(x='Gender', data=df, ax=axes[0])
sns.countplot(x='Category', data=df, ax=axes[1])
axes[1].tick_params(axis='x', rotation=45)
sns.countplot(x='City', data=df, ax=axes[2])
axes[2].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
sns.boxplot(x='Profit_Category', y='Sales', data=df)
plt.title("Sales by Profit Category")
plt.show()


In [1]:
sns.boxplot(x='Category', y='Profit', data=df)
plt.xticks(rotation=45)
plt.title("Profit by Category")
plt.show()


NameError: name 'sns' is not defined

In [ ]:
sns.pairplot(df[['Sales', 'Profit', 'Discount', 'Profit_Category']], hue='Profit_Category')
plt.show()


## Step 4: Data Cleaning
Order: remove duplicates -> fix invalid values -> handle missing values -> convert dates -> validate categoricals.
(Invalid-value correction is done **before** imputation so that values we are about to null-out get imputed sensibly, rather than imputed-then-corrupted.)


In [ ]:
# 4.1 Remove duplicates
before = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"Removed {before - df.shape[0]} duplicate rows")


In [ ]:
# 4.2 FIX: correct invalid values (present in the raw data but never handled in the original notebook)
# Qty should never be negative -> treat as missing, impute later
df.loc[df['Qty'] < 0, 'Qty'] = np.nan

# Discount should never be negative -> treat as missing, impute later
df.loc[df['Discount'] < 0, 'Discount'] = np.nan

# Customer_Age of 0 or absurdly high (>100) is not a realistic customer age -> treat as missing
df.loc[(df['Customer_Age'] <= 0) | (df['Customer_Age'] > 100), 'Customer_Age'] = np.nan

print("Invalid values converted to NaN:")
print(df[['Qty', 'Discount', 'Customer_Age']].isnull().sum())


In [ ]:
# 4.3 Handle missing values
num_cols = df.select_dtypes(include=[np.number]).columns.drop('Profit_Category')
cat_cols = df.select_dtypes(exclude=[np.number]).columns.drop(['Order_ID', 'Order_Date'])

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining missing values:", df.isnull().sum().sum())


In [ ]:
# 4.4 Convert date column
df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='%d-%m-%y', errors='coerce')
df['Order_Date'].isnull().sum()  # any dates that failed to parse


In [ ]:
# 4.5 Validate categorical values
print("Category values:", df['Category'].unique())
print("Gender values:", df['Gender'].unique())
print("City values:", df['City'].unique())


## Step 5: Outlier Detection

In [ ]:
Q1 = df['Profit'].quantile(0.25)
Q3 = df['Profit'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['Profit'] < Q1 - 1.5 * IQR) | (df['Profit'] > Q3 + 1.5 * IQR)]
print(f"Profit outliers (IQR method): {len(outliers)} rows")


In [ ]:
sns.boxplot(x='Category', y='Profit', data=df)
plt.xticks(rotation=45)
plt.title("Profit by Category (before outlier removal)")
plt.show()


In [ ]:
def remove_outliers_iqr(data, columns):
    data_clean = data.copy()
    for col in columns:
        Q1 = data_clean[col].quantile(0.25)
        Q3 = data_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        data_clean = data_clean[(data_clean[col] >= lower_bound) & (data_clean[col] <= upper_bound)]
    return data_clean

numeric_cols_for_outliers = ['Sales', 'Profit', 'Discount', 'Customer_Age', 'Rating']
print("Shape before outlier removal:", df.shape)
df = remove_outliers_iqr(df, numeric_cols_for_outliers)  # FIX: cleaned frame is now actually used downstream
df = df.reset_index(drop=True)
print("Shape after outlier removal:", df.shape)


In [ ]:
sns.boxplot(x='Category', y='Profit', data=df)
plt.xticks(rotation=45)
plt.title("Profit by Category (after outlier removal)")
plt.show()


## Step 6: Feature Engineering

In [ ]:
df['Month'] = df['Order_Date'].dt.month
df['Year'] = df['Order_Date'].dt.year
df['DayOfWeek'] = df['Order_Date'].dt.dayofweek
df['Weekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

df['ProfitMargin'] = np.where(df['Sales'] > 0, df['Profit'] / df['Sales'], 0)
df['RevenuePerItem'] = np.where(df['Qty'] > 0, df['Sales'] / df['Qty'], 0)

df[['Month', 'Year', 'DayOfWeek', 'Weekend', 'ProfitMargin', 'RevenuePerItem']].head()


## Step 7: Encoding

In [ ]:
# FIX: the original notebook assigned via `df.loc[:, col] = le.fit_transform(...)`
# on a column that already existed as dtype "object". That silently kept the column
# as object dtype (confirmed by the original notebook's own printed df.dtypes output),
# which later breaks model.fit() with a "could not convert string to float" error.
# Direct column assignment + explicit astype(int) guarantees a numeric dtype.

le_cat = LabelEncoder()
df['Category'] = le_cat.fit_transform(df['Category']).astype(int)

le_gender = LabelEncoder()
df['Gender'] = le_gender.fit_transform(df['Gender']).astype(int)

# One-Hot Encoding for City
df = pd.get_dummies(df, columns=['City'], drop_first=True)
city_dummy_cols = [c for c in df.columns if c.startswith('City_')]
df[city_dummy_cols] = df[city_dummy_cols].astype(int)

print(df.dtypes)


## Step 8 & 9: Train-Test Split and Feature Scaling
FIX: In the original notebook, `StandardScaler`/`MinMaxScaler` were fit on the **full** dataset
*before* the split (data leakage — test-set information leaks into the scaler), and the scaled
arrays were computed but never actually used by any model. Here the split happens first, and the
scaler is fit **only on the training data**, then applied to both train and test.


In [ ]:
X = df.drop(['Profit_Category', 'Order_ID', 'Order_Date'], axis=1)
y = df['Profit_Category']

feature_columns = X.columns.tolist()  # remember exact column order for later use (Step 17)
print("Feature columns:", feature_columns)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(X_train.shape, X_test.shape)


In [ ]:
num_features_to_scale = ['Sales', 'Profit', 'Discount', 'Customer_Age', 'Rating',
                          'ProfitMargin', 'RevenuePerItem']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_features_to_scale] = scaler.fit_transform(X_train[num_features_to_scale])
X_test_scaled[num_features_to_scale] = scaler.transform(X_test[num_features_to_scale])

# Distance/gradient-based models (LogReg, KNN, SVM) use the scaled features.
# Tree-based models (Decision Tree, Random Forest, Gradient Boosting) are scale-invariant,
# so they are trained on the unscaled X_train/X_test for interpretability of splits/importances.


## Step 10: Train Multiple Models
FIX: In the original notebook, the cell that *used* `models` (`for name, model in models.items(): ...`)
appeared **before** the cell that *defined* `models`, causing a `NameError` that halted the entire
rest of the notebook (confirmed from the notebook's own saved cell-execution state). The definition
and the training loop are merged into a single cell below.


In [ ]:
models = {
    "LogReg": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(probability=True, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE)
}

scaled_models = {"LogReg", "KNN", "SVM"}  # these use the scaled features

results = {}
for name, model in models.items():
    if name in scaled_models:
        model.fit(X_train_scaled, y_train)
        acc = model.score(X_test_scaled, y_test)
    else:
        model.fit(X_train, y_train)
        acc = model.score(X_test, y_test)
    results[name] = acc
    print(f"{name}: Test Accuracy = {acc:.4f}")


## Step 11: Cross Validation (K-Fold)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, model in models.items():
    Xc = X_train_scaled if name in scaled_models else X_train
    scores = cross_val_score(model, Xc, y_train, cv=cv, scoring='accuracy')
    cv_results[name] = scores.mean()
    print(f"{name}: CV Mean Accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})")


In [ ]:
pd.Series(cv_results).sort_values(ascending=False).plot(kind='bar', title='CV Mean Accuracy by Model')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.show()


## Step 12: Hyperparameter Tuning (GridSearchCV)
Tuning the two strongest tree-based ensembles from the CV comparison above (Random Forest and Gradient Boosting).


In [ ]:
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE), rf_param_grid, cv=5, n_jobs=-1)
rf_grid.fit(X_train, y_train)

print("Random Forest — Best Params:", rf_grid.best_params_)
print("Random Forest — Best CV Score:", rf_grid.best_score_)
print("Random Forest — Test Accuracy (before tuning):", results['RandomForest'])
print("Random Forest — Test Accuracy (after tuning):", rf_grid.best_estimator_.score(X_test, y_test))


In [ ]:
gb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [2, 3, 4]
}

gb_grid = GridSearchCV(GradientBoostingClassifier(random_state=RANDOM_STATE), gb_param_grid, cv=5, n_jobs=-1)
gb_grid.fit(X_train, y_train)

print("Gradient Boosting — Best Params:", gb_grid.best_params_)
print("Gradient Boosting — Best CV Score:", gb_grid.best_score_)
print("Gradient Boosting — Test Accuracy (before tuning):", results['GradientBoosting'])
print("Gradient Boosting — Test Accuracy (after tuning):", gb_grid.best_estimator_.score(X_test, y_test))


In [ ]:
# Pick whichever tuned model performs better on the test set as the final model
final_model = rf_grid.best_estimator_ if rf_grid.best_estimator_.score(X_test, y_test) >= \
              gb_grid.best_estimator_.score(X_test, y_test) else gb_grid.best_estimator_
print("Final model selected:", type(final_model).__name__)


## Step 13: Overfitting vs Underfitting
FIX: this comparison was entirely missing from the original notebook. Below, train vs. test accuracy
is compared for every model so we can flag which ones overfit (train >> test), underfit (both low),
or generalize well (train ≈ test, both reasonably high).


In [ ]:
bias_variance = []
for name, model in models.items():
    Xtr = X_train_scaled if name in scaled_models else X_train
    Xte = X_test_scaled if name in scaled_models else X_test
    train_acc = model.score(Xtr, y_train)
    test_acc = model.score(Xte, y_test)
    gap = train_acc - test_acc
    verdict = "Overfitting" if gap > 0.08 else ("Underfitting" if train_acc < 0.75 else "Good fit")
    bias_variance.append({'Model': name, 'Train Acc': train_acc, 'Test Acc': test_acc,
                           'Gap': gap, 'Verdict': verdict})

bias_variance_df = pd.DataFrame(bias_variance).sort_values('Gap', ascending=False)
bias_variance_df


In [ ]:
bias_variance_df.set_index('Model')[['Train Acc', 'Test Acc']].plot(kind='bar', figsize=(9, 5))
plt.title('Train vs Test Accuracy per Model')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Step 14: Model Evaluation (final tuned model)

In [ ]:
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:   ", recall_score(y_test, y_pred))
print("F1 Score: ", f1_score(y_test, y_pred))
print("ROC-AUC:  ", roc_auc_score(y_test, y_proba))
print()
print("Classification Report:\n", classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Low Profit', 'High Profit']).plot(cmap='Blues')
plt.title(f"Confusion Matrix — {type(final_model).__name__}")
plt.show()


## Step 15: Feature Importance
FIX: the original notebook only plotted feature importance for the single tuned Random Forest.
The spec asks for Decision Tree, Random Forest, and Gradient Boosting.


In [ ]:
tree_models_for_importance = {
    "DecisionTree": models["DecisionTree"],
    "RandomForest": rf_grid.best_estimator_,
    "GradientBoosting": gb_grid.best_estimator_
}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (name, model) in zip(axes, tree_models_for_importance.items()):
    importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(10)
    sns.barplot(x=importances.values, y=importances.index, ax=ax)
    ax.set_title(f"{name} — Top 10 Features")
plt.tight_layout()
plt.show()


## Step 16: Save Model
FIX: the original notebook only saved the model. Step 17 (predicting new data) needs the same
label encoders, scaler, and exact feature column order used at training time — those are saved too.


In [ ]:
joblib.dump(final_model, "profit_predictor.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(le_cat, "le_category.pkl")
joblib.dump(le_gender, "le_gender.pkl")
joblib.dump(feature_columns, "feature_columns.pkl")
joblib.dump(num_features_to_scale, "scaled_feature_names.pkl")
joblib.dump(type(final_model).__name__ in [t for t in ["LogReg", "KNN", "SVM"]], "final_model_uses_scaling.pkl")

print("Saved: profit_predictor.pkl, scaler.pkl, le_category.pkl, le_gender.pkl, feature_columns.pkl, scaled_feature_names.pkl")


## Step 17: Predict New Data
Every step of the original raw -> features -> encoded -> scaled -> predict pipeline is repeated
here exactly as it was applied to the training data, using the saved encoders/scaler so the new
record is transformed identically.


In [ ]:
# 17.1 Create a new raw order exactly as it would arrive before dispatch
new_order = pd.DataFrame({
    'Customer_Age': [30],
    'Gender': ['Male'],
    'City': ['Delhi'],
    'Category': ['Electronics'],
    'Qty': [3],
    'Unit Price': [5000],
    'Discount': [10],
    'Shipping': [150],
    'Delivery': [4],
    'Sales': [15000],
    'Profit': [2200],
    'Rating': [4.5],
    'Order_Date': ['15-07-26']
})
print("Step 1 - Raw new order:")
display(new_order)


In [ ]:
# 17.2 Same date-derived feature engineering as training
new_order['Order_Date'] = pd.to_datetime(new_order['Order_Date'], format='%d-%m-%y')
new_order['Month'] = new_order['Order_Date'].dt.month
new_order['Year'] = new_order['Order_Date'].dt.year
new_order['DayOfWeek'] = new_order['Order_Date'].dt.dayofweek
new_order['Weekend'] = new_order['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)
new_order['ProfitMargin'] = np.where(new_order['Sales'] > 0, new_order['Profit'] / new_order['Sales'], 0)
new_order['RevenuePerItem'] = np.where(new_order['Qty'] > 0, new_order['Sales'] / new_order['Qty'], 0)
print("Step 2 - After feature engineering:")
display(new_order[['Month', 'Year', 'DayOfWeek', 'Weekend', 'ProfitMargin', 'RevenuePerItem']])


In [ ]:
# 17.3 Apply the SAME fitted label encoders (unseen categories would raise an error here,
# which is the correct/safe behaviour rather than guessing an encoding)
le_cat_loaded = joblib.load("le_category.pkl")
le_gender_loaded = joblib.load("le_gender.pkl")

new_order['Category'] = le_cat_loaded.transform(new_order['Category'])
new_order['Gender'] = le_gender_loaded.transform(new_order['Gender'])
print("Step 3 - After label encoding Category/Gender:")
display(new_order[['Category', 'Gender']])


In [ ]:
# 17.4 One-hot encode City, then align columns to the exact training-time feature set
# (any city dummy not present in this single record is filled with 0; Order_ID/Order_Date are dropped)
new_order = pd.get_dummies(new_order, columns=['City'], drop_first=False)
# drop the base city column dummy if it exists, to mirror drop_first=True used at training time,
# then reindex to the saved training feature_columns so column order/presence matches exactly
feature_columns_loaded = joblib.load("feature_columns.pkl")
new_order = new_order.reindex(columns=feature_columns_loaded, fill_value=0)
print("Step 4 - After one-hot encoding + column alignment to training schema:")
display(new_order)


In [ ]:
# 17.5 Apply the SAME fitted scaler to the numeric columns, only if the final model needs it
scaler_loaded = joblib.load("scaler.pkl")
scaled_feature_names_loaded = joblib.load("scaled_feature_names.pkl")
uses_scaling = joblib.load("final_model_uses_scaling.pkl")

new_order_for_model = new_order.copy()
if uses_scaling:
    new_order_for_model[scaled_feature_names_loaded] = scaler_loaded.transform(new_order[scaled_feature_names_loaded])
print("Step 5 - Ready for the model (scaled)" if uses_scaling else "Step 5 - Ready for the model (tree model, no scaling needed)")


In [ ]:
# 17.6 Predict
model_loaded = joblib.load("profit_predictor.pkl")
pred_class = model_loaded.predict(new_order_for_model)[0]
pred_proba = model_loaded.predict_proba(new_order_for_model)[0]

label_map = {0: "Low Profit", 1: "High Profit"}
print("Step 6 - Prediction")
print(f"Predicted Profit_Category: {pred_class} -> {label_map[pred_class]}")
print(f"Confidence: Low Profit={pred_proba[0]:.2%}, High Profit={pred_proba[1]:.2%}")


## Step 18: Build Streamlit App
The cell below writes a `streamlit_app.py` file to disk. Run it with `streamlit run streamlit_app.py`
(it loads the same `profit_predictor.pkl`, `scaler.pkl`, encoders, and `feature_columns.pkl` saved in Step 16,
so it reproduces the exact preprocessing pipeline from Step 17).


In [ ]:
streamlit_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import joblib

st.set_page_config(page_title="ShopKart Profit Predictor", page_icon="\U0001F4E6")

@st.cache_resource
def load_artifacts():
    model = joblib.load("profit_predictor.pkl")
    scaler = joblib.load("scaler.pkl")
    le_cat = joblib.load("le_category.pkl")
    le_gender = joblib.load("le_gender.pkl")
    feature_columns = joblib.load("feature_columns.pkl")
    scaled_feature_names = joblib.load("scaled_feature_names.pkl")
    uses_scaling = joblib.load("final_model_uses_scaling.pkl")
    return model, scaler, le_cat, le_gender, feature_columns, scaled_feature_names, uses_scaling

model, scaler, le_cat, le_gender, feature_columns, scaled_feature_names, uses_scaling = load_artifacts()

st.title("ShopKart Order Profitability Predictor")
st.write("Estimate whether an order will be High Profit or Low Profit before dispatch.")

with st.form("order_form"):
    col1, col2 = st.columns(2)
    with col1:
        customer_age = st.number_input("Customer Age", 18, 90, 30)
        gender = st.selectbox("Gender", list(le_gender.classes_))
        city = st.selectbox("City", [c.replace("City_", "") for c in feature_columns if c.startswith("City_")])
        category = st.selectbox("Category", list(le_cat.classes_))
        qty = st.number_input("Quantity", 1, 20, 1)
        unit_price = st.number_input("Unit Price", 0.0, 500000.0, 1000.0)
    with col2:
        discount = st.number_input("Discount", 0.0, 500.0, 10.0)
        shipping = st.number_input("Shipping Cost", 0.0, 2000.0, 150.0)
        delivery = st.number_input("Delivery Days", 1, 30, 5)
        sales = st.number_input("Sales Amount", 0.0, 1000000.0, 5000.0)
        profit = st.number_input("Profit Amount", -100000.0, 500000.0, 500.0)
        rating = st.slider("Rating", 1.0, 5.0, 4.0)
    order_date = st.date_input("Order Date")
    submitted = st.form_submit_button("Predict Profit Category")

if submitted:
    row = pd.DataFrame({
        "Customer_Age": [customer_age], "Gender": [gender], "City": [city], "Category": [category],
        "Qty": [qty], "Unit Price": [unit_price], "Discount": [discount], "Shipping": [shipping],
        "Delivery": [delivery], "Sales": [sales], "Profit": [profit], "Rating": [rating],
        "Order_Date": [pd.to_datetime(order_date)]
    })

    row["Month"] = row["Order_Date"].dt.month
    row["Year"] = row["Order_Date"].dt.year
    row["DayOfWeek"] = row["Order_Date"].dt.dayofweek
    row["Weekend"] = row["DayOfWeek"].apply(lambda x: 1 if x >= 5 else 0)
    row["ProfitMargin"] = np.where(row["Sales"] > 0, row["Profit"] / row["Sales"], 0)
    row["RevenuePerItem"] = np.where(row["Qty"] > 0, row["Sales"] / row["Qty"], 0)

    row["Category"] = le_cat.transform(row["Category"])
    row["Gender"] = le_gender.transform(row["Gender"])

    row = pd.get_dummies(row, columns=["City"])
    row = row.reindex(columns=feature_columns, fill_value=0)

    if uses_scaling:
        row[scaled_feature_names] = scaler.transform(row[scaled_feature_names])

    pred = model.predict(row)[0]
    proba = model.predict_proba(row)[0]
    label = {0: "Low Profit", 1: "High Profit"}[pred]

    st.subheader(f"Prediction: {label}")
    st.write(f"Low Profit probability: {proba[0]:.1%}")
    st.write(f"High Profit probability: {proba[1]:.1%}")
'''

with open("streamlit_app.py", "w") as f:
    f.write(streamlit_code)

print("streamlit_app.py written to disk")


## Step 19: requirements.txt

In [ ]:
requirements = '''pandas
numpy
matplotlib
seaborn
scikit-learn
joblib
streamlit
'''

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt written to disk")


## Step 20: Deployment Instructions

**Local run**
1. Ensure `profit_predictor.pkl`, `scaler.pkl`, `le_category.pkl`, `le_gender.pkl`,
   `feature_columns.pkl`, `scaled_feature_names.pkl`, `final_model_uses_scaling.pkl`,
   `streamlit_app.py`, and `requirements.txt` are all in the same folder.
2. `pip install -r requirements.txt`
3. `streamlit run streamlit_app.py`
4. Open the URL Streamlit prints (default `http://localhost:8501`).

**Deploy to Streamlit Community Cloud**
1. Push this folder (app + `.pkl` artifacts + `requirements.txt`) to a GitHub repo.
2. Go to streamlit.io/cloud, sign in, and click "New app".
3. Point it at the repo/branch and set the main file to `streamlit_app.py`.
4. Deploy — the app rebuilds automatically on every push to the branch.

**Notes**
- Re-run Steps 1–16 and re-save the `.pkl` artifacts whenever the model is retrained, so the
  Streamlit app and the training pipeline never drift out of sync.
- If new cities or categories appear in future data, the saved `LabelEncoder`s will raise an
  error on unseen labels — that is intentional, so a mismatch is caught rather than silently
  mis-encoded. Retrain and re-save the encoders when the category set changes.
